# Structured text and line media - Rust

All 8 Rust examples from [docs/text.md](https://platob.github.io/yggdryl/text/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

## Text media and Arrow batches

In [ ]:
use yggdryl::generic::IORecordOptions;
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::Url;

fn named(name: &str) -> yggdryl::Result<Buffer> {
    Ok(Buffer::new().with_media_type(Url::from_str(&format!("file:///{name}"))?.media_type()))
}

let mut source = named("app.log")?;
source.write_all_bytes(b"first event\nsecond event\n")?;

let options = source.record_options()?;
let rows: usize = source
    .read_arrow_reader(&options)?
    .map(|batch| batch.map(|batch| batch.num_rows()))
    .sum::<Result<_, _>>()?;
assert_eq!(rows, 2);

let mut target = named("copy.log")?;
target.overwrite_arrow_reader(source.read_arrow_reader(&options)?, &options)?;
target.append_arrow_reader(source.read_arrow_reader(&options)?, &options)?;
assert_eq!(
    target.read_all_bytes()?,
    b"first event\nsecond event\nfirst event\nsecond event\n"
);

let merging = options.clone().with_merge_by_names(["message"]);
let refused = target.merge_arrow_reader(source.read_arrow_reader(&options)?, &merging);
assert!(refused.unwrap_err().to_string().contains("row identity"));

### Line iteration with `Text`

In [ ]:
use yggdryl::io::{Buffer, IOBase};

let handle = Buffer::from_bytes(b"first event\nsecond event\n".to_vec()).into_text();
let mut records = handle.read_lines()?;

assert_eq!(records.next().unwrap()?.text()?, "first event");
assert_eq!(records.next().unwrap()?.text()?, "second event");
assert!(records.next().is_none());

## Raw shared-Value access

In [ ]:
use yggdryl::{json, Value};

let quote = json::from_utf8(r#"{"symbol":"AAPL","price":12.5}"#)?;

assert_eq!(
    quote.get_key_str("symbol").and_then(Value::as_utf8),
    Some("AAPL")
);
assert_eq!(json::into_utf8(&quote)?, r#"{"price":12.5,"symbol":"AAPL"}"#);

### Typed `Value` families

In [ ]:
use yggdryl::Value;

assert_eq!(
    Value::I8(-1).checked_add(&Value::U8(2))?,
    Value::I16(1),
);
assert_eq!(
    Value::d128(1, 0).checked_div(&Value::d128(2, 0))?,
    Value::d128(5, 1),
);

## Field-directed parsing

In [ ]:
use yggdryl::{json, DataType, Field, Value};

let amount = Field::new(
    "amount",
    DataType::decimal128(8, 2)?,
    false,
);
let value = json::from_utf8_with_field(r#""12.50""#, &amount)?;

assert_eq!(value, Value::d128(1_250, 2));

## Raw document codecs

In [ ]:
use yggdryl::text::{self, Format};
use yggdryl::Value;

let (format, value) = text::from_utf8_inferred(r#"{"id":1}"#)?;

assert_eq!(format, Format::Json);
assert_eq!(value.get_key_str("id"), Some(&Value::U64(1)));
assert_eq!(text::into_utf8(&value, format)?, r#"{"id":1}"#);

## Formatting

In [ ]:
use yggdryl::text::Formatting;
use yggdryl::{json, Value};

let value = Value::from_record([("id", Value::I64(1))])?;
let pretty =
    json::into_utf8_with_formatting(&value, Formatting::indented(2))?;

assert_eq!(pretty, "{\n  \"id\": 1\n}");
assert_eq!(json::from_utf8(&pretty)?, value);

## Placeholders

In [ ]:
use yggdryl::text::{Format, Loading, Placeholders};
use yggdryl::Value;

let loading = Loading::new().with_placeholders(
    Placeholders::new().with_variable("HOST", Value::from("db.internal")),
);
let value = yggdryl::text::from_utf8_with(
    "host: \"{{ HOST }}\"\nport: \"{{ PORT | default(8080) }}\"\n",
    Format::Yaml,
    &loading,
)?;

assert_eq!(value.get_key_str("host").and_then(Value::as_utf8), Some("db.internal"));
assert_eq!(value.get_key_str("port"), Some(&Value::I64(8080)));